In [ ]:
import pandas as pd
import numpy as np

## Colunas que iremos trabalhar no dataset:<br>
**contador: identificador do registro<br>**
**TIPOOBITO: se é um óbito fetal ou não<br>**
**DTOBITO: data em que ocorreu o óbito<br>**
**DTNASC: data de nascimento<br>**
**IDADE: campo que mostra em minutos, horas, dias ou anos<br>**

In [ ]:
# 1. Define a lista de colunas que serão importadas
colunas_desejadas = [
    'contador',
    'TIPOBITO',
    'DTOBITO',
    'DTNASC',
    'IDADE',
    'SEXO',
    'RACACOR',
    'ESTCIV',
    'ESC2010',
    'OCUP',
    'CODMUNRES',
    'CODMUNOCOR',
    'CAUSABAS',
    'ACIDTRAB'
]

# 2. Defina os tipos de dados das colunas
tipos_dados = {
    'TIPOBITO': str,
    'IDADE': str,       # Mantém como texto para não perder os zeros à esquerda da codificação do DATASUS
    'SEXO': str,
    'RACACOR': str,
    'ESTCIV': str,
    'ESC2010': str,
    'OCUP': str,
    'CODMUNRES': int,
    'CODMUNOCOR': str,
    'CAUSABAS': str     # Garante que códigos CID mistos não gerem alertas
}

# 3. Importa o arquivo CSV de forma otimizada
mortalidade = pd.read_csv(
    'Mortalidade_Geral_2026.csv',
    sep=';',
    usecols=colunas_desejadas,
    dtype=tipos_dados,
    parse_dates=['DTOBITO', 'DTNASC'],
    date_format='%d/%m/%Y'
)

# 4. Converte DTOBITO e DTNASC (formato: ddmmYYYY) para uma data real do Pandas (formato: YYYY-mm-dd)
mortalidade['DTOBITO'] = pd.to_datetime(mortalidade['DTOBITO'], format='%d%m%Y', errors='coerce')
mortalidade['DTNASC'] = pd.to_datetime(mortalidade['DTNASC'], format='%d%m%Y', errors='coerce')

In [ ]:
mortalidade.head()

In [ ]:
mortalidade.info()

In [ ]:
municipios = pd.read_excel('MUNICIPIOS.xlsx')

In [ ]:
municipios.head()

In [ ]:
municipios.info()

**Abaixo feito um merge com a tabela de municipios para poder analisar os dados de cada municipio**

In [ ]:
mortalidade = mortalidade.merge(municipios, how='left')

In [ ]:
mortalidade.head()

In [ ]:
mortalidade.info()

In [ ]:
cid = pd.read_excel('CIDs_com_descricao.xlsx')

In [ ]:
cid.head()

**feito um merge com a tabela de cids, para poder identificar as maiores causas de morte em 2026**

In [ ]:
mortalidade = mortalidade.merge(cid, how='left', left_on='CAUSABAS', right_on='CAUSABAS')

In [ ]:
mortalidade.head()

In [ ]:
mortalidade.info()

In [ ]:
mortalidade['CAUSABAS'].isna().sum()

In [ ]:
mortalidade['DESCRICAO_CID'].isna().sum()

**cids que não foram localizados na tabela oficial**

In [ ]:
mortalidade.loc[mortalidade['DESCRICAO_CID'].isna(), ['CAUSABAS', 'DESCRICAO_CID']]

In [ ]:
cids_faltantes = pd.DataFrame(cid.loc[cid['DESCRICAO_CID'].isna(), ['CAUSABAS', 'DESCRICAO_CID']])

In [ ]:
cids_faltantes.to_csv('CIDs_faltantes.csv', index=False)

In [ ]:
cid.loc[cid['CAUSABAS']=='A090', ['CAUSABAS', 'DESCRICAO_CID']]

**10 maiores causas de morte em 2026:**

In [ ]:
mortalidade['DESCRICAO_CID'].value_counts().head(10)

**10 maiores municipios em quantidade de mortes em 2026:**

In [ ]:
mortalidade['MUNICIPIO'].value_counts().head(10)

In [ ]:
acidentes = mortalidade[mortalidade['ACIDTRAB']== 1.0]

In [ ]:
acidentes.head()

**10 maiores causas de morte por acidentes de trabalho em 2026:**

In [ ]:
acidentes['DESCRICAO_CID'].value_counts().head(10)